<a href="https://colab.research.google.com/github/SlavaMarina/ib-cs-ml-course/blob/main/week2_supervised/Week2_Lesson4_Workshop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏦 IB CS — Неделя 2 · Урок 4 (Семинар)
## Workshop "The Loan Predictor" — Kaggle-style Competition
### A4.3.2 (Classification) + A4.3.3 (Hyperparameter tuning) + A4.2.2 (Feature selection в действии)

> ⏱ Длительность: 2 академических часа.
> 💻 Платформа: Google Colab
> 🎯 Цель: пройти полный цикл **обучения классификатора** в стиле Kaggle: данные → фичи → модель → метрики → тюнинг.

---

### 📋 План семинара

| Часть | Тема | Время |
|---|---|---|
| 1 | Загрузка датасета "Loan Predictor" | 10 мин |
| 2 | Feature engineering & selection (A4.2.2) | 15 мин |
| 3 | Train/Test split | 10 мин |
| 4 | Базовая KNN модель | 15 мин |
| 5 | Базовая Decision Tree модель | 15 мин |
| 6 | Расчёт всех 4 метрик + Confusion Matrix | 20 мин |
| 7 | 🏆 Соревнование: hyperparameter tuning | 25 мин |
| 8 | IB-style вывод | 10 мин |

---

### ⚔️ Соревнование

В конце урока вы покажете свой **лучший F1-score**. Победитель получает **+5 бонусных баллов** к финальному ДЗ. 😎


## Часть 1 · Датасет "Loan Predictor"

> Контекст: банк решает, **одобрить** или **отказать** в кредите. Мы тренируем модель, чтобы автоматизировать это решение.


In [ ]:
# === Генерация синтетического датасета ===
# ВАЖНО: одинаковый для всех студентов (фиксированный seed)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
np.random.seed(2027)
sns.set_style('whitegrid')

n = 1000
df = pd.DataFrame({
    'age':              np.random.normal(40, 12, n).clip(18, 75).round(0),
    'annual_income_k':  np.random.lognormal(3.8, 0.6, n).clip(15, 300).round(1),
    'employment_years': np.random.exponential(8, n).clip(0, 40).round(1),
    'credit_score':     np.random.normal(680, 80, n).clip(300, 850).round(0),
    'debt_to_income':   np.random.beta(2, 5, n).round(3),  # 0-1
    'loan_amount_k':    np.random.lognormal(3.2, 0.7, n).clip(5, 200).round(1),
    'num_credit_cards': np.random.choice([0,1,2,3,4,5,6,7], n,
                                          p=[0.05,0.2,0.3,0.2,0.1,0.08,0.05,0.02]),
    'years_education':  np.random.choice([10,12,14,16,18], n,
                                          p=[0.1,0.4,0.15,0.25,0.1]),
    'random_noise':     np.random.normal(0, 1, n),  # шум — должен быть отброшен
})

# Целевая переменная: одобрен ли кредит?
# Логика: высокий credit_score + хороший доход + низкий долг + умеренная сумма кредита
score = (
    0.4 * (df['credit_score'] - 600) / 250
    + 0.25 * np.log1p(df['annual_income_k']) / np.log(300)
    - 0.3 * df['debt_to_income']
    - 0.15 * df['loan_amount_k'] / df['annual_income_k'].clip(20)
    + 0.1 * df['employment_years'] / 40
    + np.random.normal(0, 0.15, n)
)
df['loan_approved'] = (score > score.median()).astype(int)

print(f"📊 Размер датасета: {df.shape}")
print(f"   Approved (1): {df['loan_approved'].sum()} ({df['loan_approved'].mean()*100:.0f}%)")
print(f"   Denied   (0): {(1-df['loan_approved']).sum()} ({(1-df['loan_approved'].mean())*100:.0f}%)")
df.head(8)

### 🎯 TRY IT #1 — Какие фичи кажутся вам важными?

Перед тем, как смотреть на корреляции, **угадайте**:
- Какие 3 фичи вы бы оставили?
- Какие 2 фичи выбросили бы?

> 💡 **Совет:** доверяйте здравому смыслу. Это финансовая модель — какие факторы реально влияют на одобрение кредита?


## Часть 2 · Feature Selection (A4.2.2) на практике

Применим **filter method** (Pearson's r) — самый частый approach для IB IA.


In [ ]:
# Корреляции с целевой переменной
correlations = df.corr()['loan_approved'].drop('loan_approved').abs().sort_values(ascending=False)
print("=== |Pearson's r| с loan_approved ===")
print(correlations.round(3))

# Визуализация
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['green' if v > 0.1 else 'orange' if v > 0.05 else 'red' for v in correlations]
ax.barh(correlations.index, correlations.values, color=colors, edgecolor='black')
ax.axvline(0.1, color='black', linestyle='--', label='Threshold = 0.1')
ax.set_xlabel("|Pearson's r|", fontsize=11)
ax.set_title('Filter Method: ранжирование фич по корреляции с целью',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout(); plt.show()

# Тепловая карта всех корреляций
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Корреляционная матрица — full picture', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

### 🎯 TRY IT #2 — Решите, какие фичи использовать

Глядя на correlations:

1. Какая фича имеет **самую высокую** корреляцию с `loan_approved`?
2. Какая фича — **шум** (близко к 0)?
3. Есть ли пары фич с высокой корреляцией **между собой** (мультиколлинеарность)?

> 💎 **IB-style обоснование:** *"I removed `random_noise` because its correlation with the target was below 0.05, indicating no predictive value. I retained the top X features based on a threshold of |r| > 0.1."*


In [ ]:
# Финальный набор фич: оставляем те, что коррелируют > 0.05
# (в реальном IA threshold обычно подбирается через cross-validation)
selected_features = correlations[correlations > 0.05].index.tolist()
print(f"✅ Оставляем {len(selected_features)} фич: {selected_features}")
print(f"❌ Отбрасываем: {[c for c in correlations.index if c not in selected_features]}")

## Часть 3 · Train/Test Split + Нормализация

> 💎 **СЕКРЕТ:** в IB IA **обязательно** должны быть отдельные train и test наборы. Оценка на тех же данных, на которых обучались = **методологическая ошибка**.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[selected_features]
y = df['loan_approved']

# 70% train, 30% test (стандарт для IB)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape}")
print(f"Test size:  {X_test.shape}")
print(f"\n⚠️ stratify=y → сохраняем пропорцию классов в train и test:")
print(f"  Train approved %: {y_train.mean()*100:.1f}%")
print(f"  Test  approved %: {y_test.mean()*100:.1f}%")

# Нормализация (StandardScaler) — критично для KNN!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # ВАЖНО: transform, НЕ fit_transform на test!

print("\n✅ Нормализация выполнена (StandardScaler)")
print("   ⚠️ На test используем .transform(), а не .fit_transform()!")

> 🚨 **КРИТИЧЕСКАЯ ОШИБКА:** `scaler.fit_transform(X_test)` — это **data leakage**. На test мы используем **те же параметры** scaler'а, что были рассчитаны на train. Это **частая ошибка в IA** — потеря 1–2 баллов.


## Часть 4 · Базовая KNN модель

> Гиперпараметр: `n_neighbors` (=K). Начнём с K=5.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

y_pred_knn = knn.predict(X_test_scaled)

# Метрики
print("=== KNN (k=5) — базовая модель ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_knn):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_knn):.4f}")

print("\n=== classification_report ===")
print(classification_report(y_test, y_pred_knn, target_names=['Denied', 'Approved']))

In [ ]:
# Confusion Matrix визуализация
cm_knn = confusion_matrix(y_test, y_pred_knn)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted: Denied', 'Predicted: Approved'],
            yticklabels=['Actual: Denied', 'Actual: Approved'],
            cbar=False, ax=ax, annot_kws={'size': 14})
ax.set_title('Confusion Matrix — KNN (k=5)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Идентифицируем TP, FP, TN, FN явно для понимания
TN, FP, FN, TP = cm_knn.ravel()
print(f"\nTN (correctly denied)    = {TN}")
print(f"FP (wrongly approved)    = {FP}  ← потенциальные плохие кредиты!")
print(f"FN (wrongly denied)      = {FN}  ← упущенные клиенты")
print(f"TP (correctly approved)  = {TP}")

## Часть 5 · Базовая Decision Tree модель

> Гиперпараметр: `max_depth`. Начнём с depth=5.
> Преимущество: **НЕ требует нормализации**.


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# ВАЖНО: для Decision Tree можно использовать НЕнормализованные данные
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)  # без scaling!

y_pred_dt = dt.predict(X_test)

print("=== Decision Tree (max_depth=5) ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1 Score:  {f1_score(y_test, y_pred_dt):.4f}")

In [ ]:
# Confusion matrix для Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Denied', 'Approved'],
            yticklabels=['Denied', 'Approved'],
            cbar=False, ax=axes[0], annot_kws={'size': 14})
axes[0].set_title('KNN (k=5)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Denied', 'Approved'],
            yticklabels=['Denied', 'Approved'],
            cbar=False, ax=axes[1], annot_kws={'size': 14})
axes[1].set_title('Decision Tree (depth=5)', fontweight='bold')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')

plt.tight_layout(); plt.show()

In [ ]:
# Визуализация самого дерева — БОЛЬШОЕ преимущество для интерпретации
fig, ax = plt.subplots(figsize=(18, 9))
plot_tree(dt, feature_names=selected_features, class_names=['Denied', 'Approved'],
          filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title('Decision Tree — interpretable model (рисует пути решений)',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Какие фичи дерево использовало?
importances = pd.Series(dt.feature_importances_, index=selected_features).sort_values(ascending=False)
print("\n=== Feature importances (по дереву) ===")
print(importances.round(3))

> 💎 **СЕКРЕТ для IA:** `feature_importances_` от Decision Tree — это **embedded method** feature selection (вспомните A4.2.2!). Можно использовать для **последующей** фильтрации признаков.


## Часть 6 · Сравнение метрик — model comparison

> 📘 Это превью **A4.3.10 — Model Selection**. На IB Section B обычно просят: *"Justify which model should be deployed."*


In [ ]:
# Сводная таблица
results = pd.DataFrame({
    'KNN (k=5)': [
        accuracy_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_knn),
    ],
    'Decision Tree (depth=5)': [
        accuracy_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_dt),
    ],
}, index=['Accuracy', 'Precision', 'Recall', 'F1 Score']).round(4)

print(results)

# Bar chart сравнения
fig, ax = plt.subplots(figsize=(10, 5))
results.plot(kind='bar', ax=ax, color=['steelblue', 'green'], edgecolor='black')
ax.set_ylabel('Score'); ax.set_ylim(0, 1.05)
ax.set_title('KNN vs Decision Tree — сравнение метрик', fontsize=12, fontweight='bold')
ax.legend(loc='lower right'); ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## 🏆 Часть 7 · СОРЕВНОВАНИЕ — Hyperparameter Tuning

> **Цель:** найти **наилучшие** гиперпараметры для KNN и Decision Tree.
> **Метрика для ранжирования:** F1 Score.

### 7.1 — KNN tuning


In [ ]:
# Перебираем разные K
k_values = list(range(1, 50, 2))  # нечётные K от 1 до 49 (избегаем tie)
f1_scores_knn = []
acc_scores_knn = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    f1_scores_knn.append(f1_score(y_test, pred))
    acc_scores_knn.append(accuracy_score(y_test, pred))

best_k = k_values[np.argmax(f1_scores_knn)]
print(f"🏆 Лучшее K = {best_k}")
print(f"   F1 = {max(f1_scores_knn):.4f}")
print(f"   Accuracy = {acc_scores_knn[np.argmax(f1_scores_knn)]:.4f}")

# График
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(k_values, f1_scores_knn, 'o-', label='F1 Score', color='steelblue', linewidth=2)
ax.plot(k_values, acc_scores_knn, 's--', label='Accuracy', color='orange', alpha=0.7)
ax.axvline(best_k, color='red', linestyle=':', label=f'Best K = {best_k}')
ax.set_xlabel('K (число соседей)', fontsize=11)
ax.set_ylabel('Score'); ax.set_title('KNN Hyperparameter Tuning', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 7.2 — Decision Tree tuning

In [ ]:
# Перебираем разные max_depth
depths = list(range(1, 21))
f1_scores_dt = []
train_acc_dt = []  # для демонстрации overfitting
test_acc_dt = []

for d in depths:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    f1_scores_dt.append(f1_score(y_test, pred_test))
    train_acc_dt.append(accuracy_score(y_train, pred_train))
    test_acc_dt.append(accuracy_score(y_test, pred_test))

best_depth = depths[np.argmax(f1_scores_dt)]
print(f"🏆 Лучшая max_depth = {best_depth}")
print(f"   F1 = {max(f1_scores_dt):.4f}")

# График + демонстрация overfitting
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(depths, f1_scores_dt, 'o-', color='green', linewidth=2)
axes[0].axvline(best_depth, color='red', linestyle=':', label=f'Best depth = {best_depth}')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('F1 Score')
axes[0].set_title('Decision Tree: F1 vs depth', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(depths, train_acc_dt, 'o-', label='Train accuracy', color='steelblue')
axes[1].plot(depths, test_acc_dt, 's-', label='Test accuracy', color='orange')
axes[1].fill_between(depths, train_acc_dt, test_acc_dt, alpha=0.2, color='red')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Overfitting visualization (gap = overfit)', fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

> 🔬 **Наблюдение:** заметили, как **train accuracy всё время растёт**, а **test accuracy перестаёт расти** после оптимальной глубины? Это **overfitting в чистом виде**. Красная область = разрыв = "память без обобщения".


## Часть 8 · Финальное сравнение и IB-style вывод


In [ ]:
# === Финальные модели с оптимальными гиперпараметрами ===
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train_scaled, y_train)
y_pred_best_knn = best_knn.predict(X_test_scaled)

best_dt = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_dt.fit(X_train, y_train)
y_pred_best_dt = best_dt.predict(X_test)

# Сводная таблица
final = pd.DataFrame({
    f'KNN (k={best_k})': [
        accuracy_score(y_test, y_pred_best_knn),
        precision_score(y_test, y_pred_best_knn),
        recall_score(y_test, y_pred_best_knn),
        f1_score(y_test, y_pred_best_knn),
    ],
    f'Decision Tree (depth={best_depth})': [
        accuracy_score(y_test, y_pred_best_dt),
        precision_score(y_test, y_pred_best_dt),
        recall_score(y_test, y_pred_best_dt),
        f1_score(y_test, y_pred_best_dt),
    ],
}, index=['Accuracy', 'Precision', 'Recall', 'F1']).round(4)

print("=" * 50)
print("🏆 ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ")
print("=" * 50)
print(final)

best_model = final.idxmax(axis=1).iloc[3]  # лучший по F1
print(f"\n🥇 Победитель по F1: {best_model}")
print(f"   F1 = {final[best_model].iloc[3]:.4f}")

### 🎯 TRY IT #3 — Запишите свой результат

> ⚔️ Ваш Best F1 Score: **____________**
>
> 🏆 Записывайте на доске — у кого выше?


### 📝 IB-style вывод (для отчёта IA)

> Это **шаблон отчёта**, который вы напишете в ДЗ. Сегодня — заполняем коротко.

```
1. PREPROCESSING (A4.2):
   - Применён filter method (Pearson's r) для feature selection.
   - Удалена фича `random_noise` (|r| < 0.05) — не несла предсказательной ценности.
   - StandardScaler применён только для KNN (Decision Tree не требует).

2. MODELS TRAINED:
   - KNN with optimal K = ___ → F1 = ___
   - Decision Tree with optimal depth = ___ → F1 = ___

3. JUSTIFICATION:
   - Decision Tree обеспечивает большую интерпретируемость (важно для банка —
     регулирующие органы требуют объяснимости решений).
   - KNN показал [больший/меньший] F1, что делает его [более/менее] подходящим.

4. RECOMMENDATION:
   Для production-системы рекомендуется ___ , так как:
   (a) ___
   (b) ___
   (c) ___
```


## ✅ Что мы сделали за семинар

- [x] **Feature selection** через filter method (A4.2.2) ✓
- [x] **Train/Test split** с stratification и нормализацией
- [x] Обучили **KNN** (lazy learner) — A4.3.2 ✓
- [x] Обучили **Decision Tree** (eager learner) — A4.3.2 ✓
- [x] Визуализировали само дерево (interpretability!)
- [x] Рассчитали **все 4 метрики** + confusion matrix — A4.3.3 ✓
- [x] Провели **hyperparameter tuning** (K и max_depth) — A4.3.3 ✓
- [x] Увидели **overfitting** на графиках train vs test
- [x] Сравнили модели через **F1 score** (preview A4.3.10)

---

### 🏠 ДЗ (см. `Week2_HW2_Practice.ipynb`)

> Полная реализация **KNN vs Decision Tree** на **другом** датасете + **IB-style отчёт Model Selection**.

---

### 💎 Финальные секреты семинара

1. **KNN без нормализации = катастрофа.** Всегда `StandardScaler` или `MinMaxScaler`.
2. **Decision Tree без `max_depth` = переобучение.** Всегда ограничивайте глубину.
3. **`fit_transform` на train, `transform` на test** — иначе data leakage = -1 балл в IA.
4. **`stratify=y`** в `train_test_split` — критично для несбалансированных классов.
5. **F1 score** > Accuracy при дисбалансе классов — всегда так пишите в отчёте.
6. **`feature_importances_`** от Decision Tree — это бесплатный feature selection (вспомните A4.2.2 embedded methods).
